# Percentile Pickle — 60/20/20 User-Level Split for MLP Tier-1 Agents

This notebook creates a deterministic **60% Train / 20% Validation / 20% Test** user-level split of the percentile pickle used by the MLP Tier-1 agents (`data/dayr4.2-percentile30.pkl`).

It replicates **exactly** the split methodology and seed from `week3/tier1_score_generation.ipynb`:
- source pickle: `data/dayr4.2-percentile30.pkl`
- user identifier column: `user`
- malicious user = user whose max `insider` label `> 0`
- seed: `random_state=42` / `SEED=42`
- split: stratified user-level 60/20/20 (Train vs Temp=40%, then Temp halves)

The original pickle is **not** modified. Outputs are saved to `data/user_split_formlp/`.

## 1. Imports and Configuration

In [1]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# === Determinism: seed all RNGs (same config as week3/tier1_score_generation.ipynb) ===
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Project root (portability convention)
_curr = Path.cwd()
while not (_curr / 'pyproject.toml').exists() and _curr != _curr.parent:
    _curr = _curr.parent
PROJECT_ROOT = _curr

DATA_DIR = PROJECT_ROOT / 'data'
SRC_PKL = DATA_DIR / 'dayr4.2-percentile30.pkl'
OUT_DIR = DATA_DIR / 'user_split_formlp'
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_TRAIN = OUT_DIR / 'dayr4.2-percentile30-train.pkl'
OUT_VAL   = OUT_DIR / 'dayr4.2-percentile30-validation.pkl'
OUT_TEST  = OUT_DIR / 'dayr4.2-percentile30-test.pkl'

print(f"SEED={SEED}")
print(f"Source pickle : {SRC_PKL}")
print(f"Output dir    : {OUT_DIR}")

SEED=42
Source pickle : C:\insider_threat_detection_project2\data\dayr4.2-percentile30.pkl
Output dir    : C:\insider_threat_detection_project2\data\user_split_formlp


## 2. Load Percentile Dataset

In [2]:
if not SRC_PKL.exists():
    raise FileNotFoundError(f"Source percentile pickle not found: {SRC_PKL}")

# Load the full percentile pickle (same file the MLP Tier-1 agents use).
df = pd.read_pickle(str(SRC_PKL))

# Downcast float64 -> float32 (same memory convention as tier1_score_generation.ipynb).
for col in df.select_dtypes(include=['float64']).columns:
    df[col] = df[col].astype('float32')

print(f"Percentile data shape: {df.shape}")

Percentile data shape: (308023, 510)


## 3. Inspect Dataset Structure

In [3]:
print(f"Columns: {len(df.columns)}")
print(f"First columns: {df.columns.tolist()[:8]} ...")
print(f"Meta columns present: {[c for c in ['user','week','day','insider'] if c in df.columns]}")
print(f"user dtype: {df['user'].dtype} | insider dtype: {df['insider'].dtype}")
print(f"Unique users: {df['user'].nunique()}")
print(f"Total rows: {len(df)}")
print("\nInsider label distribution (per row):")
print(df['insider'].value_counts().sort_index().to_string())

Columns: 510
First columns: ['isweekday', 'isweekend', 'n_allact', 'allact_n-pc0', 'allact_n-pc1', 'allact_n-pc2', 'allact_n-pc3', 'n_workhourallact'] ...
Meta columns present: ['user', 'week', 'day', 'insider']
user dtype: float32 | insider dtype: float32
Unique users: 1000
Total rows: 308023

Insider label distribution (per row):
insider
0.0    307057
1.0        85
2.0       861
3.0        20


## 4. Identify Users and Malicious Users

A **user is malicious** if their maximum `insider` label is `> 0` (exactly how `tier1_score_generation.ipynb` identifies them via `df.groupby('user')['insider'].max()`).

In [4]:
# Per-user max insider label (malicious if > 0), exactly as tier1_score_generation.ipynb
user_labels = df.groupby('user')['insider'].max().astype(int)
unique_users = user_labels.index.values

malicious_mask = user_labels > 0
total_users = len(unique_users)
total_malicious = int(malicious_mask.sum())

print(f"Total unique users   : {total_users}")
print(f"Total malicious users: {total_malicious}")
print(f"Total normal users   : {total_users - total_malicious}")
print("\nMalicious-user label breakdown:")
print(user_labels[malicious_mask].value_counts().sort_index().to_string())

Total unique users   : 1000
Total malicious users: 70
Total normal users   : 930

Malicious-user label breakdown:
insider
1    30
2    30
3    10


## 5. Reproducible 60/20/20 User Split

In [5]:
# === Exact split logic from week3/tier1_score_generation.ipynb ===
# 1) 60% Train vs 40% Temp (stratified by per-user insider label)
train_users, temp_users = train_test_split(
    unique_users, test_size=0.4, random_state=SEED, stratify=user_labels.values)

# 2) Temp split into two equal halves -> 20% Validation, 20% Test
val_users, test_users = train_test_split(
    temp_users, test_size=0.5, random_state=SEED,
    stratify=user_labels.loc[temp_users].values)

train_users = np.asarray(train_users)
val_users = np.asarray(val_users)
test_users = np.asarray(test_users)

print(f"Train users: {len(train_users)} | Validation users: {len(val_users)} | Test users: {len(test_users)}")

Train users: 600 | Validation users: 200 | Test users: 200


## 6. Split Statistics

In [6]:
train_df = df[df['user'].isin(train_users)].copy()
val_df   = df[df['user'].isin(val_users)].copy()
test_df  = df[df['user'].isin(test_users)].copy()

def user_stats(users):
    ul = user_labels.loc[users]
    return len(users), int((ul > 0).sum()), int((ul == 0).sum())

tr_u, tr_m, tr_n = user_stats(train_users)
va_u, va_m, va_n = user_stats(val_users)
te_u, te_m, te_n = user_stats(test_users)

print(f"{'Split':<12}{'Total Users':>13}{'Malicious':>12}{'Normal':>10}{'Rows':>10}")
print('-' * 57)
print(f"{'Train':<12}{tr_u:>13}{tr_m:>12}{tr_n:>10}{len(train_df):>10}")
print(f"{'Validation':<12}{va_u:>13}{va_m:>12}{va_n:>10}{len(val_df):>10}")
print(f"{'Test':<12}{te_u:>13}{te_m:>12}{te_n:>10}{len(test_df):>10}")
print(f"{'Total':<12}{total_users:>13}{total_malicious:>12}{total_users-total_malicious:>10}{len(df):>10}")

print("\nMalicious-user rate per split:")
print(f"  Train      = {100*tr_m/tr_u:.2f}%")
print(f"  Validation = {100*va_m/va_u:.2f}%")
print(f"  Test       = {100*te_m/te_u:.2f}%")

Split         Total Users   Malicious    Normal      Rows
---------------------------------------------------------
Train                 600          42       558    185744
Validation            200          14       186     61076
Test                  200          14       186     61203
Total                1000          70       930    308023

Malicious-user rate per split:
  Train      = 7.00%
  Validation = 7.00%
  Test       = 7.00%


## 7. Save Split Files

In [7]:
train_df.to_pickle(str(OUT_TRAIN))
val_df.to_pickle(str(OUT_VAL))
test_df.to_pickle(str(OUT_TEST))

print(f"Saved:\n  {OUT_TRAIN}\n  {OUT_VAL}\n  {OUT_TEST}")
print(f"\nSource pickle untouched: {SRC_PKL}")

Saved:
  C:\insider_threat_detection_project2\data\user_split_formlp\dayr4.2-percentile30-train.pkl
  C:\insider_threat_detection_project2\data\user_split_formlp\dayr4.2-percentile30-validation.pkl
  C:\insider_threat_detection_project2\data\user_split_formlp\dayr4.2-percentile30-test.pkl

Source pickle untouched: C:\insider_threat_detection_project2\data\dayr4.2-percentile30.pkl


## 8. Verify No User Leakage

In [8]:
# Reload the saved files and verify user leakage + row integrity.
tr2 = pd.read_pickle(str(OUT_TRAIN))
va2 = pd.read_pickle(str(OUT_VAL))
te2 = pd.read_pickle(str(OUT_TEST))

t2, v2, te2u = set(tr2['user'].unique()), set(va2['user'].unique()), set(te2['user'].unique())
ov_tv = len(t2 & v2)
ov_tte = len(t2 & te2u)
ov_vte = len(v2 & te2u)

print(f"Train/Validation overlap: {ov_tv}")
print(f"Train/Test overlap:       {ov_tte}")
print(f"Validation/Test overlap:  {ov_vte}")
assert ov_tv == 0 and ov_tte == 0 and ov_vte == 0, "USER LEAKAGE DETECTED!"

total_reloaded = len(tr2) + len(va2) + len(te2)
print(f"\nRow integrity: {len(tr2)} + {len(va2)} + {len(te2)} = {total_reloaded} (original {len(df)})")
assert total_reloaded == len(df), "ROW COUNT MISMATCH!"
print("[OK] No user leakage; all rows accounted for.")

Train/Validation overlap: 0
Train/Test overlap:       0
Validation/Test overlap:  0

Row integrity: 185744 + 61076 + 61203 = 308023 (original 308023)
[OK] No user leakage; all rows accounted for.


## 9. Verify Malicious Users in Each Split

In [9]:
# Recompute per-user max insider on the reloaded files and confirm the
# malicious-user counts match the split-time counts.
def count_malicious(part):
    return int((part.groupby('user')['insider'].max() > 0).sum())

m_tr = count_malicious(tr2)
m_va = count_malicious(va2)
m_te = count_malicious(te2)

print(f"Malicious users -> Train: {m_tr} | Validation: {m_va} | Test: {m_te}")
print(f"Expected       -> Train: {tr_m} | Validation: {va_m} | Test: {te_m}")
assert (m_tr, m_va, m_te) == (tr_m, va_m, te_m), "Malicious-user count mismatch!"
print("[OK] Malicious users correctly distributed per split.")

Malicious users -> Train: 42 | Validation: 14 | Test: 14
Expected       -> Train: 42 | Validation: 14 | Test: 14
[OK] Malicious users correctly distributed per split.


## 10. Reproducibility Check

In [10]:
# Re-run the split with the same seed and confirm identical user assignments.
np.random.seed(SEED)
train_users_b, temp_users_b = train_test_split(
    unique_users, test_size=0.4, random_state=SEED, stratify=user_labels.values)
val_users_b, test_users_b = train_test_split(
    temp_users_b, test_size=0.5, random_state=SEED,
    stratify=user_labels.loc[temp_users_b].values)

train_identical = np.array_equal(np.sort(train_users), np.sort(np.asarray(train_users_b)))
val_identical   = np.array_equal(np.sort(val_users),   np.sort(np.asarray(val_users_b)))
test_identical  = np.array_equal(np.sort(test_users),  np.sort(np.asarray(test_users_b)))

print(f"Train users identical:      {train_identical}")
print(f"Validation users identical: {val_identical}")
print(f"Test users identical:       {test_identical}")
assert train_identical and val_identical and test_identical, "Split is NOT reproducible!"
print("[OK] Split is deterministic/reproducible with SEED=42.")

Train users identical:      True
Validation users identical: True
Test users identical:       True
[OK] Split is deterministic/reproducible with SEED=42.


## 11. Final Summary

In [11]:
print("=" * 60)
print("PERCENTILE DATASET - MLP USER SPLIT")
print("=" * 60)
print(f"Source: {SRC_PKL}")
print("\nSplit:")
print("  Train      : 60%")
print("  Validation : 20%")
print("  Test       : 20%")
print("\nUsers:")
print(f"  Total      : {total_users}")
print(f"  Train      : {tr_u}")
print(f"  Validation : {va_u}")
print(f"  Test       : {te_u}")
print("\nMalicious Users:")
print(f"  Total      : {total_malicious}")
print(f"  Train      : {tr_m}")
print(f"  Validation : {va_m}")
print(f"  Test       : {te_m}")
print("\nRows:")
print(f"  Total      : {len(df)}")
print(f"  Train      : {len(train_df)}")
print(f"  Validation : {len(val_df)}")
print(f"  Test       : {len(test_df)}")
print("\nUser Leakage:")
print(f"  Train <-> Validation : {ov_tv}")
print(f"  Train <-> Test       : {ov_tte}")
print(f"  Validation <-> Test  : {ov_vte}")
print("\nReproducibility:")
print(f"  Train identical      : {train_identical}")
print(f"  Validation identical : {val_identical}")
print(f"  Test identical       : {test_identical}")
print(f"\nFiles saved to: {OUT_DIR}")
print("=" * 60)
print("Split completed successfully")
print("No user leakage")
print("Original dataset preserved")
print("Split is reproducible")
print("=" * 60)

PERCENTILE DATASET - MLP USER SPLIT
Source: C:\insider_threat_detection_project2\data\dayr4.2-percentile30.pkl

Split:
  Train      : 60%
  Validation : 20%
  Test       : 20%

Users:
  Total      : 1000
  Train      : 600
  Validation : 200
  Test       : 200

Malicious Users:
  Total      : 70
  Train      : 42
  Validation : 14
  Test       : 14

Rows:
  Total      : 308023
  Train      : 185744
  Validation : 61076
  Test       : 61203

User Leakage:
  Train <-> Validation : 0
  Train <-> Test       : 0
  Validation <-> Test  : 0

Reproducibility:
  Train identical      : True
  Validation identical : True
  Test identical       : True

Files saved to: C:\insider_threat_detection_project2\data\user_split_formlp
Split completed successfully
No user leakage
Original dataset preserved
Split is reproducible
